In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://www.tenderadvisor.com/IndianTenders/Keyword-Tenders/online-tenders-for-di-pipe/0/1"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

tenders = []
for tender_div in soup.find_all('div', class_='tender-list'):
    ref_no = sector = status = tender_type = desc = location = due_date = amount = ''
    bid_link = eligibility_link = gem_link = ''

    ref_tag = tender_div.find('span', class_='ref-id')
    if ref_tag:
        ref_no = ref_tag.get_text(strip=True).replace('Ref. No', '').replace('#', '').strip()
    sector_tag = tender_div.find('span', class_='sector')
    if sector_tag:
        sector = sector_tag.contents[0].strip()
        badge = sector_tag.find('small')
        if badge:
            status = badge.get_text(strip=True)
    tender_type_tag = tender_div.find('small', class_='badge bg-orange')
    if tender_type_tag:
        tender_type = tender_type_tag.get_text(strip=True)
    desc_tag = tender_div.find('p')
    if desc_tag and desc_tag.a:
        desc = desc_tag.a.get_text(" ", strip=True)
        notice_url = desc_tag.a['href']
        base_url = 'https://www.tenderadvisor.com'
        bid_link = base_url + notice_url.replace('Tender-Notice', 'Bidding-Tender')
        eligibility_link = base_url + notice_url.replace('Tender-Notice', 'Check-Eligibility-Criteria')
        gem_link = base_url + notice_url.replace('Tender-Notice', 'GeM-Registration')
    info_tags = tender_div.find_all('div', class_='single-info')
    for info in info_tags:
        heading = info.find('h5').get_text(strip=True)
        val = info.find('span').get_text(strip=True)
        if 'Location' in heading:
            location = val
        elif 'Due Date' in heading:
            due_date = val
        elif 'Amount' in heading:
            amount = val

    tenders.append({
        'Ref. No': ref_no,
        'Sector': sector,
        'Status': status,
        'Type': tender_type,
        'Description': desc,
        'Location': location,
        'Due Date': due_date,
        'Amount': amount,
        'Bid Link': bid_link,
        'Eligibility Link': eligibility_link,
        'GeM Link': gem_link
    })

df = pd.DataFrame(tenders)
df.to_excel('tenderadvisor_di_pipe_tenders_live.xlsx', index=False)
print('Scraping complete. Data saved to tenderadvisor_di_pipe_tenders_live.xlsx')


Scraping complete. Data saved to tenderadvisor_di_pipe_tenders_live.xlsx


In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Target URL
url = "https://www.tenderadvisor.com/IndianTenders/Keyword-Tenders/online-tenders-for-di-pipe/0/1"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Will stop the script if the request fails

soup = BeautifulSoup(response.text, 'lxml')

# --- Extract Breadcrumb (Category/Keyword) ---
breadcrumb = soup.find('ol', class_='breadcrumb')
if breadcrumb:
    breadcrumb_items = [li.get_text(" ", strip=True) for li in breadcrumb.find_all('li')]
    breadcrumb_str = " > ".join(breadcrumb_items)
else:
    breadcrumb_str = ""

# --- Extract Active Filters ---
keywords_wrap = soup.find('div', class_='keywords-wrap')
if keywords_wrap:
    filters_str = keywords_wrap.get_text(" ", strip=True)
else:
    filters_str = ""

# --- Scrape All Tenders on the Page ---
tenders = []
for tender_div in soup.find_all('div', class_='tender-list'):
    ref_no = sector = status = tender_type = desc = location = due_date = amount = ''
    bid_link = eligibility_link = gem_link = ''

    # Ref No
    ref_tag = tender_div.find('span', class_='ref-id')
    if ref_tag:
        ref_no = ref_tag.get_text(strip=True).replace('Ref. No', '').replace('#', '').strip()

    # Sector and Status
    sector_tag = tender_div.find('span', class_='sector')
    if sector_tag:
        sector = sector_tag.contents[0].strip()
        badge = sector_tag.find('small')
        if badge:
            status = badge.get_text(strip=True)
    tender_type_tag = tender_div.find('small', class_='badge bg-orange')
    if tender_type_tag:
        tender_type = tender_type_tag.get_text(strip=True)

    # Description & Links
    desc_tag = tender_div.find('p')
    if desc_tag and desc_tag.a:
        desc = desc_tag.a.get_text(" ", strip=True)
        notice_url = desc_tag.a['href']
        base_url = 'https://www.tenderadvisor.com'
        bid_link = base_url + notice_url.replace('Tender-Notice', 'Bidding-Tender')
        eligibility_link = base_url + notice_url.replace('Tender-Notice', 'Check-Eligibility-Criteria')
        gem_link = base_url + notice_url.replace('Tender-Notice', 'GeM-Registration')

    # Location, Due Date, Amount
    info_tags = tender_div.find_all('div', class_='single-info')
    for info in info_tags:
        heading = info.find('h5').get_text(strip=True)
        val = info.find('span').get_text(strip=True)
        if 'Location' in heading:
            location = val
        elif 'Due Date' in heading:
            due_date = val
        elif 'Amount' in heading:
            amount = val

    tenders.append({
        'Breadcrumb': breadcrumb_str,
        'Active Filters': filters_str,
        'Ref. No': ref_no,
        'Sector': sector,
        'Status': status,
        'Type': tender_type,
        'Description': desc,
        'Location': location,
        'Due Date': due_date,
        'Amount': amount,
        'Bid Link': bid_link,
        'Eligibility Link': eligibility_link,
        'GeM Link': gem_link
    })

# --- Export to Excel ---
df = pd.DataFrame(tenders)
df.to_excel('tenderadvisor_di_pipe_tenders_live1.xlsx', index=False)
print('Scraping complete. Data saved to tenderadvisor_di_pipe_tenders_live1.xlsx')

# (Optional) Print what you scraped:
print("Breadcrumb:", breadcrumb_str)
print("Active Filters:", filters_str)
print(f"Total tenders scraped: {len(tenders)}")


Scraping complete. Data saved to tenderadvisor_di_pipe_tenders_live1.xlsx
Breadcrumb: Home > Indian Tenders > Keyword Tenders > di pipe Tenders
Active Filters: 
Total tenders scraped: 20
